In [0]:
from pyspark.sql import functions as F
# Aggregate chargers by FSA
df_charger_supply = (spark.table("ev_spark.silver.alt_fuel_stns")
                        .groupBy("fsa")
                        .agg(F.count("Station").alias("num_stations"),
                             F.sum("EV_L1_Count").alias("basic_chargers"),
                             F.sum("EV_L2_Count").alias("normal_chargers"),
                             F.sum("EV_L3_Count").alias("fast_chargers"))
                        .withColumn("total_chargers", F.col("basic_chargers") + F.col("normal_chargers") + F.col("fast_chargers"))
)

df_ev_combined = spark.table("ev_spark.silver.ev_combined")

df_gold_gap = df_ev_combined.join(df_charger_supply, on="fsa", how="left").fillna(0)
# Calculate the Gap Metric: Chargers per 1,000 EVs
df_gold_gap = (df_gold_gap
                    .withColumn("chargers_per_1k_evs", 
                                F.when(F.col("ev_count")>0,
                                        F.round((F.col("total_chargers") / F.col("ev_count")) * 1000,2)
                                        ).otherwise(None)
                    )
                    )

# Identify "Priority Zones"
# If the ratio is below 5 (industry benchmark), mark as a gap 
df_gold_gap = (df_gold_gap
                    .withColumn("Latitude", F.col("Latitude").cast("double"))
                    .withColumn("Longitude", F.col("Longitude").cast("double"))
                    .withColumn("gap_priority", 
                                F.when(F.col("chargers_per_1k_evs") < 5, "High")
                                .when(F.col("chargers_per_1k_evs") < 10, "Medium")
                                .otherwise("Low")
                    )
                    )


In [0]:
df_gold_gap.write.mode("overwrite").saveAsTable("ev_spark.gold.gold_ev_infrastructure_gap")

In [0]:
%sql
-- drop table if exists ev_spark.gold.gold_ev_infrastructure_gap
select * from ev_spark.gold.gold_ev_infrastructure_gap

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.